In [ ]:
"""
Apply temporal-backtest uncertainty bands to SSP projections.
Produces:
  - 4_ssp_with_uncertainty_bands.csv   (per scenario × year × waste)
  - 4_ssp_global_total_summary.csv     (annual total + error for paper text)
"""
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import curve_fit

# ── Locate release root ──
def find_release_root(start: Path) -> Path:
    sentinel_dirs = ('Step0_Data', 'Step9_FutureProjection')
    for candidate in [start, *start.parents]:
        if all((candidate / name).is_dir() for name in sentinel_dirs):
            return candidate
        nested_project = candidate / 'Global-solid-waste-prediction'
        if all((nested_project / name).is_dir() for name in sentinel_dirs):
            return nested_project
    raise FileNotFoundError('Could not locate Global-solid-waste-prediction repository root')


release = find_release_root(Path.cwd().resolve())
step8_dir = release / 'Step9_FutureProjection' / '2_Results'


In [ ]:
# ── Step 1: Fit error-vs-horizon curve ──
err_data = pd.read_csv(step8_dir / '3_global_error_vs_horizon.csv')

# Fit sqrt model for each waste type + total
def sqrt_model(x, a):
    return a * np.sqrt(x)


error_params = {}
for wt in ['AW', 'CDW', 'IW', 'MSW', 'Total_SW']:
    sub = err_data[err_data['WasteType'] == wt].copy()
    h = sub['Horizon'].values.astype(float)
    wape = sub['Mean_Abs_Rel_Error_Pct'].values / 100
    popt, _ = curve_fit(sqrt_model, h, wape, p0=[0.05], maxfev=5000)
    error_params[wt] = float(popt[0])
    print(f'{wt}: WAPE(h) = {popt[0]:.6f} * sqrt(h)')

print('\nProjected Total_SW WAPE:')
a_total = error_params['Total_SW']
for yr in [2030, 2050, 2080, 2100]:
    h = yr - 2024
    print(f'  {yr} (h={h}): +/-{sqrt_model(h, a_total)*100:.1f}%')


In [ ]:
# ── Step 2: Load super panel and aggregate to global totals ──
panel = pd.read_csv(step8_dir / '2_super_panel_1990_2100.csv')

# Aggregate to global totals per scenario × year × waste type
records = []
for scenario in sorted(panel['Scenario'].unique()):
    for year in sorted(panel['Year'].unique()):
        mask = (panel['Scenario'] == scenario) & (panel['Year'] == year)
        sub = panel[mask]
        if len(sub) == 0:
            continue
        row = {'Scenario': scenario, 'Year': year}
        total = 0.0
        for wt in ['AW', 'CDW', 'IW', 'MSW']:
            val = sub[f'{wt}_final'].sum()
            row[f'{wt}_Gt'] = val / 1e9
            total += val
        row['Total_SW_Gt'] = total / 1e9
        records.append(row)

global_df = pd.DataFrame(records)
print(f'Global aggregation: {len(global_df)} rows')
print(global_df[global_df['Scenario'] == 'SSP2'][['Year', 'Total_SW_Gt']].tail(10).to_string(index=False))


In [ ]:
# ── Step 3: Apply uncertainty bands ──
ANCHOR_YEAR = 2024  # The finalize model was trained up to this year

bands_rows = []
for _, row in global_df.iterrows():
    year = int(row['Year'])
    scenario = row['Scenario']
    
    # For historical years (<= anchor), use horizon=0 conceptually.
    # But we still have model-filled gaps, so use a minimum floor error.
    if year <= ANCHOR_YEAR:
        # For historical: error comes from gap-filling.
        # Use the 1-year backtest WAPE as a conservative proxy.
        horizon = max(1, ANCHOR_YEAR - year + 1)
        # Cap at the maximum observed backtest horizon for historical
        horizon = min(horizon, 19)  
    else:
        # For future: distance from anchor year
        horizon = year - ANCHOR_YEAR
    
    new_row = {'Scenario': scenario, 'Year': year}
    
    for wt in ['AW', 'CDW', 'IW', 'MSW', 'Total_SW']:
        val_gt = row[f'{wt}_Gt']
        a = error_params[wt]
        wape_frac = sqrt_model(horizon, a)
        
        # For historical years where most data is observed, 
        # scale down uncertainty (not all countries are gap-filled)
        if year <= ANCHOR_YEAR:
            # Rough estimate: ~30% of data points are gap-filled for historical
            gap_fill_fraction = 0.15  
            effective_wape = wape_frac * gap_fill_fraction
        else:
            effective_wape = wape_frac
        
        new_row[f'{wt}_Gt'] = val_gt
        new_row[f'{wt}_WAPE_pct'] = effective_wape * 100
        new_row[f'{wt}_Upper_Gt'] = val_gt * (1 + effective_wape)
        new_row[f'{wt}_Lower_Gt'] = max(0, val_gt * (1 - effective_wape))
        new_row[f'{wt}_Error_Gt'] = val_gt * effective_wape
    
    bands_rows.append(new_row)

bands_df = pd.DataFrame(bands_rows)
bands_path = step8_dir / '4_ssp_with_uncertainty_bands.csv'
bands_df.to_csv(bands_path, index=False)
print(f'\nSaved: {bands_path} ({len(bands_df)} rows)')


In [ ]:
# ── Step 4: Generate summary table for paper text ──
# Key years for paper reporting
KEY_YEARS = [2000, 2010, 2019, 2024, 2030, 2040, 2050, 2060, 2070, 2080, 2090, 2100]
SCENARIOS = ['SSP1', 'SSP2', 'SSP3', 'SSP4', 'SSP5']

summary_rows = []
for year in KEY_YEARS:
    for scenario in SCENARIOS:
        row_mask = (bands_df['Year'] == year) & (bands_df['Scenario'] == scenario)
        if row_mask.sum() == 0:
            continue
        r = bands_df[row_mask].iloc[0]
        summary_rows.append({
            'Year': year,
            'Scenario': scenario,
            'Total_SW_Gt': round(r['Total_SW_Gt'], 2),
            'Total_SW_Error_Gt': round(r['Total_SW_Error_Gt'], 2),
            'Total_SW_Lower_Gt': round(r['Total_SW_Lower_Gt'], 2),
            'Total_SW_Upper_Gt': round(r['Total_SW_Upper_Gt'], 2),
            'Total_SW_WAPE_pct': round(r['Total_SW_WAPE_pct'], 1),
            'AW_Gt': round(r['AW_Gt'], 2),
            'CDW_Gt': round(r['CDW_Gt'], 2),
            'IW_Gt': round(r['IW_Gt'], 2),
            'MSW_Gt': round(r['MSW_Gt'], 2),
        })

summary_df = pd.DataFrame(summary_rows)
summary_path = step8_dir / '4_ssp_global_total_summary.csv'
summary_df.to_csv(summary_path, index=False)
print(f'Saved: {summary_path}')


In [ ]:
# ── Print paper-ready table ──
print('\n' + '=' * 90)
print('GLOBAL TOTAL SOLID WASTE (Gt) WITH UNCERTAINTY — Key Years')
print('=' * 90)

# Historical (same across all scenarios)
hist_years = [y for y in KEY_YEARS if y <= 2024]
print('\n--- Historical ---')
print(f'{"Year":>6}  {"Total_SW":>12}  {"±Error":>10}  {"Range":>22}  {"WAPE%":>6}')
for year in hist_years:
    r = summary_df[(summary_df['Year'] == year) & (summary_df['Scenario'] == 'SSP1')].iloc[0]
    print(f'{year:>6}  {r["Total_SW_Gt"]:>10.2f} Gt  {r["Total_SW_Error_Gt"]:>8.2f} Gt  [{r["Total_SW_Lower_Gt"]:.2f} - {r["Total_SW_Upper_Gt"]:.2f}]  {r["Total_SW_WAPE_pct"]:>5.1f}%')

# Future by scenario
fut_years = [y for y in KEY_YEARS if y > 2024]
print('\n--- Future Projections ---')
header = f'{"Year":>6}'
for sc in SCENARIOS:
    header += f'  {sc:>20}'
print(header)

for year in fut_years:
    line = f'{year:>6}'
    for sc in SCENARIOS:
        match = summary_df[(summary_df['Year'] == year) & (summary_df['Scenario'] == sc)]
        if len(match) > 0:
            r = match.iloc[0]
            line += f'  {r["Total_SW_Gt"]:.1f}±{r["Total_SW_Error_Gt"]:.1f} Gt'
        else:
            line += f'  {"N/A":>20}'
    print(line)
